In [8]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Polygon
import re
from pathlib import Path

# 1. Load the Data
gdb_path = r"data\raw\alameda_parcel_ownership_20260915.gdb"
gdf = gpd.read_file(gdb_path, layer="PARCEL_With_Ownership")

# 2. Define Categorization Functions
def categorize_ownership(row):
    use_code = str(row.get('usecode', '')).strip()
    owner = str(row.get('ownername', '')).upper()
    
    # Public Entities
    if use_code in ['300', '6001', '6100']:
        if 'UNITED STATES' in owner or 'US ' in owner: return 'Public - Federal'
        if 'STATE OF CA' in owner: return 'Public - State'
        if 'COUNTY' in owner: return 'Public - County'
        if 'CITY' in owner or 'OAKLAND' in owner: return 'Public - City'
        return 'Public - Other'
        
    # Corporate & Trust (Overrides standard residential if owned by entity)
    if re.search(r'\b(LLC|INC|CORP|LTD|LP)\b', owner):
        return 'Corporate/Investor'
    if re.search(r'\b(TRUST|TR)\b', owner):
        return 'Trust'
        
    # Communal / Condo
    if use_code.startswith('73') or use_code.startswith('15') or use_code.startswith('16') or use_code == '3900' or use_code == '4101':
        return 'Condo/Communal'
        
    # Individual / Single Family
    if use_code == '1100' or use_code.startswith('2'):
        return 'Individual/Small Multi-Family'
        
    return 'Other/Commercial'

# Apply categorization
gdf['owner_category'] = gdf.apply(categorize_ownership, axis=1)

# 3. Create the Northern Waterfront Boundary using Shapely
# Defining the bounding box using approximate WGS84 coordinates
nw_polygon_wgs84 = Polygon([
    (-122.277, 37.771), # NW Corner (Webster at Estuary)
    (-122.250, 37.771), # NE Corner (Willow at Estuary)
    (-122.250, 37.763), # SE Corner (Willow at Central)
    (-122.277, 37.763)  # SW Corner (Webster at Central)
])

# Create a GeoDataFrame for the boundary and project it to the parcel CRS (EPSG:2227)
boundary_gdf = gpd.GeoDataFrame(index=[0], crs='EPSG:4326', geometry=[nw_polygon_wgs84])
boundary_gdf = boundary_gdf.to_crs(gdf.crs)

# 4. Spatial Join/Flagging
# Create a boolean column marking if a parcel intersects the Northern Waterfront boundary
gdf['in_northern_waterfront'] = gdf.intersects(boundary_gdf.geometry.iloc[0])

# 5. Clean up missing values and write to GeoParquet
gdf['yearbuilt'] = gdf['yearbuilt'].replace(0, pd.NA)
gdf['buildingarea'] = gdf['buildingarea'].replace(0, pd.NA)

output_name = "alameda_parcels_cleaned.parquet"
output_path = Path("data") / "cleaned" / output_name
output_path.parent.mkdir(parents=True, exist_ok=True)
output_name = "alameda_parcels_cleaned.parquet"
gdf.to_parquet(output_path, index=False)
print(f"Data successfully cleaned and exported to {output_path}")

Data successfully cleaned and exported to data\cleaned\alameda_parcels_cleaned.parquet


In [9]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# Load the cleaned data
gdf = gpd.read_parquet("alameda_parcels_cleaned.parquet")

# 1. Stakeholder Breakdown
# Compare the makeup of the Northern Waterfront vs the rest of Alameda
stats = gdf.groupby(['in_northern_waterfront', 'owner_category']).size().unstack(fill_value=0)
stats.index = ['Rest of Alameda', 'Northern Waterfront']
print("Parcel Counts by Ownership Category:")
print(stats.T)

# 2. Visual Preparation
# Keep only one polygon per building footprint for clean rendering
map_gdf = gdf.drop_duplicates(subset=['geometry'])

# 3. Build the Map
fig, ax = plt.subplots(figsize=(14, 10))

# Plot the rest of the island with a muted opacity to serve as context
map_gdf[~map_gdf['in_northern_waterfront']].plot(
    ax=ax, 
    column='owner_category', 
    cmap='Set3', 
    alpha=0.3,
    linewidth=0.1,
    edgecolor='white'
)

# Plot the Northern Waterfront fully opaque
map_gdf[map_gdf['in_northern_waterfront']].plot(
    ax=ax, 
    column='owner_category', 
    cmap='Set1', 
    alpha=1.0,
    linewidth=0.2,
    edgecolor='black',
    legend=True,
    legend_kwds={'title': 'Ownership Category', 'bbox_to_anchor': (1, 1)}
)

ax.set_title("Stakeholder Distribution: Northern Waterfront Sea-Level Rise Adaptation", fontsize=16)
ax.set_axis_off()
plt.tight_layout()
plt.show()

FileNotFoundError: [WinError 2] Failed to open local file 'alameda_parcels_cleaned.parquet'. Detail: [Windows error 2] The system cannot find the file specified.
